# UNSW-NB15: Feature Engineering

In [75]:
import os, pickle, gc  # files, saving objects, freeing memory
import numpy as np  # for array
import pandas as pd  # for csv files and dataframe
import matplotlib.pyplot as plt  # for plotting
import seaborn as sns  # plotting
from scipy import stats

In [76]:
# Loading data from disk
train = pd.read_csv('../data/train_alldata_EDA.csv')
test = pd.read_csv('../data/test_alldata_EDA.csv')


C:\Users\ENPEI\AppData\Local\Temp\ipykernel_96068\2103487528.py:2: DtypeWarning: Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../data/train_alldata_EDA.csv')
C:\Users\ENPEI\AppData\Local\Temp\ipykernel_96068\2103487528.py:3: DtypeWarning: Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv('../data/test_alldata_EDA.csv')


In [77]:
train.shape

(1778032, 49)

In [78]:
# Utility function
def multi_corr(col1, col2="label", df=train):
    '''
    This function returns correlation between 2 given features.
    Also gives corr of the given features with "label" afetr applying log1p to it.
    '''
    corr = df[[col1, col2]].corr().iloc[0,1]
    log_corr = df[col1].apply(np.log1p).corr(df[col2])

    print("Correlation : {}\nlog_Correlation: {}".format(corr, log_corr))

In [79]:
def corr(col1, col2="label", df=train):
    """
    This function returns correlation between 2 given features
    """
    return df[[col1, col2]].corr().iloc[0,1]

## Removing highly correlated features

In [80]:
corr_matrix = train.corr(numeric_only=True).abs()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(np.bool))

# Find index of feature columns with correlation greater than 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

In [81]:
# We don't want to use these features for plotting because these are having high corr
# And most likely have same kind of plots with already plotted feature
print(to_drop)

['sloss', 'dloss', 'dpkts', 'dwin', 'ltime', 'ct_srv_dst', 'ct_src_dport_ltm', 'ct_dst_src_ltm']


In [82]:
# Reuse the dictionary saved by the first notebook (it already holds 'columns', 'binary_col', ...)
old_file = './final_ipynb'

if os.path.isfile(old_file):
    saved_dict = pickle.load(open(old_file, 'rb'))
else:
    saved_dict = {}
    print("WARNING: ./final_ipynb not found, starting with an empty saved_dict")

print(list(saved_dict.keys()))   # should contain 'columns' and 'binary_col'
saved_dict['corr_col'] = to_drop  # add the new key on top of the old ones

['columns', 'binary_col']


In [83]:
# removing the features from train and test data
train.drop(columns=to_drop, inplace=True)

In [84]:
test.drop(columns=to_drop, inplace=True)

In [85]:
train.shape, test.shape

((1778032, 41), (762015, 41))

## Adding New Features

Refer: https://www.elastic.co/guide/en/ecs/master/ecs-network.html
* Network bytes: Total bytes trasferred by the network. It is sum of 'sbytes' (Source to destination bytes) and 'dbytes' (Destination to source bytes).

In [86]:
# creating new features
train['network_bytes'] = train['sbytes'] + train['dbytes']

In [87]:
train.shape, test.shape

((1778032, 42), (762015, 41))

In [88]:
# Dropping columns which are not useful for the classification
# attack_cat is for multiclass classification
# all the other columns are address related and not present in sample train data
train.drop(['srcip', 'sport', 'dstip', 'dsport', 'attack_cat'], axis=1, inplace=True)

In [89]:
# To use during test data transformation
saved_dict['to_drop'] = ['srcip', 'sport', 'dstip', 'dsport', 'attack_cat']

In [90]:
train.shape, test.shape

((1778032, 37), (762015, 41))

In [91]:
train["service"]=train["service"].fillna("others")


In [92]:
train["service"].value_counts()

service
others      872286
dns         547094
http        144555
ftp-data     88310
smtp         57160
ftp          34293
ssh          32897
pop3          1076
dhcp           124
ssl            109
snmp            81
radius          27
irc             20
Name: count, dtype: int64

## Applying log1p on Numerical columns

During EDA we found that few numerical columns shows better visualization for pdf curves if we apply log1p to the columns.

So I thought to try log1p on all the columns and check the correlation value of the original column and log1p column with target column i.e. "label"

In [93]:
# Getting number of unique values of all the columns
# If the unique values are high that means it has continuous set of values
col_unique_values = train.nunique()
col_unique_values

proto                   135
state                    16
dur                  458708
sbytes                11619
dbytes                15519
sttl                     13
dttl                     11
service                  13
sload                813944
dload                857308
spkts                   857
swin                     22
stcpb               1007023
dtcpb               1006615
smeansz                1402
dmeansz                1408
trans_depth              12
res_bdy_len            2352
sjit                 978134
djit                1013756
stime                 85317
sintpkt              728686
dintpkt              719690
tcprtt                58566
synack                52389
ackdat                47665
is_sm_ips_ports           2
ct_state_ttl              7
ct_flw_http_mthd         16
is_ftp_login              2
ct_ftp_cmd                8
ct_srv_src               65
ct_dst_ltm               62
ct_src_ltm               62
ct_dst_sport_ltm         55
label               

In [94]:
# If the unique values are getter than some threshould than we will check its corr
col = col_unique_values[col_unique_values>200].index

In [95]:
# Checking corr value of original col and log1p applied col
# Taking those columns whose unique values are getter than some threshould
for column in col:
    print("{:-^30}".format(column))
    multi_corr(column)

-------------dur--------------
Correlation : 0.0019274028701131475
log_Correlation: -0.0325441375646063
------------sbytes------------


Correlation : 0.010344749695229565
log_Correlation: -0.3561631555898472
------------dbytes------------
Correlation : -0.07641408324436148
log_Correlation: -0.5193868283741491
------------sload-------------
Correlation : 0.19211948100086756
log_Correlation: 0.34746601450349374
------------dload-------------
Correlation : -0.21978094390126515
log_Correlation: -0.6033545881626396
------------spkts-------------
Correlation : -0.12200425437154418
log_Correlation: -0.3163533826967591
------------stcpb-------------
Correlation : -0.23365153315010911
log_Correlation: -0.31355632221429003
------------dtcpb-------------
Correlation : -0.23346071773809843
log_Correlation: -0.31340064798121114
-----------smeansz------------
Correlation : -0.06517990378993671
log_Correlation: -0.1511145098964839
-----------dmeansz------------
Correlation : -0.27230605607442226
log_Correlation: -0.5646402127401746
---------res_bdy_len----------
Correlation : -0.0268968526601601
log_Correlation: -0.06794901821505812


In [96]:
# Will apply log1p on this columns and remove original columns
log1p_col = ['dur', 'sbytes', 'dbytes', 'sload', 'dload', 'spkts', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'sjit', 'djit', 'network_bytes']

In [97]:
saved_dict['log1p_col'] = log1p_col

In [98]:
# mode values of every features, will use to fill Null values of test
mode_dict = train.mode().iloc[0].to_dict()
mode_dict

{'proto': 'tcp',
 'state': 'FIN',
 'dur': 9e-06,
 'sbytes': 146,
 'dbytes': 0,
 'sttl': 31,
 'dttl': 29,
 'service': 'others',
 'sload': 50666664.0,
 'dload': 0.0,
 'spkts': 2,
 'swin': 255,
 'stcpb': 0,
 'dtcpb': 0,
 'smeansz': 132,
 'dmeansz': 0,
 'trans_depth': 0,
 'res_bdy_len': 0,
 'sjit': 0.0,
 'djit': 0.0,
 'stime': 1421927443,
 'sintpkt': 0.009,
 'dintpkt': 0.0,
 'tcprtt': 0.0,
 'synack': 0.0,
 'ackdat': 0.0,
 'is_sm_ips_ports': 0,
 'ct_state_ttl': 0,
 'ct_flw_http_mthd': 0.0,
 'is_ftp_login': 0,
 'ct_ftp_cmd': 0,
 'ct_srv_src': 1,
 'ct_dst_ltm': 2,
 'ct_src_ltm': 2,
 'ct_dst_sport_ltm': 1,
 'label': 0,
 'network_bytes': 324}

In [99]:
def log1p_transform(col, df=train):
    '''
    Apply log1p on given column.
    Remove the original cola and keep log1p applied col
    '''
    new_col = col+'_log1p'
    df[new_col] = df[col].apply(np.log1p)
    df.drop(col, axis=1, inplace=True)

In [100]:
# Transforming columns with log1p
for col in log1p_col:
    log1p_transform(col, df=train)

In [101]:
train.shape

(1778032, 37)

In [102]:
train.columns

Index(['proto', 'state', 'sttl', 'dttl', 'service', 'swin', 'trans_depth',
       'res_bdy_len', 'stime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack',
       'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd',
       'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_dst_sport_ltm', 'label', 'dur_log1p', 'sbytes_log1p',
       'dbytes_log1p', 'sload_log1p', 'dload_log1p', 'spkts_log1p',
       'stcpb_log1p', 'dtcpb_log1p', 'smeansz_log1p', 'dmeansz_log1p',
       'sjit_log1p', 'djit_log1p', 'network_bytes_log1p'],
      dtype='object')

In [103]:
train.shape, test.shape

((1778032, 37), (762015, 41))

In [104]:
# creating x and y set from the dataset
x_train, y_train = train.drop(columns=['label']), train['label']
x_test, y_test = test.drop(columns=['label']), test['label']

In [105]:
print(x_train.shape, y_train.shape)
print()
print(x_test.shape, y_test.shape)

(1778032, 36) (1778032,)

(762015, 40) (762015,)


In [106]:
# Final model-ready train/test pickles are saved after both splits are preprocessed.

In [107]:
# getting categorical and numerical columns in 2 diff lists
cat_col = ['proto', 'service', 'state']
num_col = [c for c in x_train.columns if c not in cat_col]  # keeps the column order

In [108]:
# To use later, during test data cleaning
saved_dict['cat_col'] = cat_col
saved_dict['num_col'] = num_col

In [109]:
x_train.head()

,proto,state,sttl,dttl,service,swin,trans_depth,res_bdy_len,stime,sintpkt,...,sload_log1p,dload_log1p,spkts_log1p,stcpb_log1p,dtcpb_log1p,smeansz_log1p,dmeansz_log1p,sjit_log1p,djit_log1p,network_bytes_log1p
0,udp,INT,254,0,others,0,0,0,1421930643,33.479000,...,9.275600,0.000000,1.609438,0.000000,0.000000,3.828641,0.000000,3.878042,0.000000,5.187386
1,udp,INT,60,0,dns,0,0,0,1424246229,0.008000,...,18.698312,0.000000,1.098612,0.000000,0.000000,4.890349,0.000000,0.000000,0.000000,5.579730
2,tcp,FIN,31,29,others,255,0,0,1421948071,0.372205,...,14.105347,16.314201,3.713572,20.196135,21.733479,4.174387,6.313548,0.000000,3.012070,10.152883
3,tcp,FIN,31,29,ftp,255,0,0,1421971944,16.144740,...,10.258074,10.501435,3.970292,21.803017,20.494420,4.043051,4.248495,7.264606,3.984562,8.806124
4,tcp,FIN,31,29,others,255,0,0,1421963050,1.218800,...,13.339317,13.412088,2.833213,20.673269,21.855078,4.574711,4.521789,4.309533,1.138118,8.066208


## Outlier Detection

Extreme values (outliers) can distort ML models. However, in network intrusion data, **outliers are often the attacks themselves** (e.g., massive byte counts in a DDoS).

We detect them to understand the data, not to blindly delete them. 

### IQR Method (Interquartile Range)
Finds values outside `Q1 - 1.5 * IQR` and `Q3 + 1.5 * IQR`. Highly robust because it uses percentiles instead of the mean.

In [110]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Function to find outlier bounds using IQR
def get_iqr_bounds(df, columns):
    Q1 = df[columns].quantile(0.25)
    Q3 = df[columns].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

# 2. Get the bounds for your numerical columns
lower_bounds, upper_bounds = get_iqr_bounds(x_train, num_col)

# 3. Detect outliers
outliers_mask = (x_train[num_col] < lower_bounds) | (x_train[num_col] > upper_bounds)

# 4. Calculate the PROPORTION (percentage) instead of the count
# .mean() on a boolean mask gives the proportion of True values
outlier_proportions = outliers_mask.mean().sort_values(ascending=False) * 100

print("--- Proportion of Outliers per Feature (IQR Method) ---")
# Print only columns that have at least 1 outlier, formatted as percentages
valid_proportions = outlier_proportions[outlier_proportions > 0]
for col, prop in valid_proportions.items():
    print(f"{col:25s} : {prop:.2f}%")

# 5. Plot Boxplots (Diagrammes en moustaches)
# n_cols = 4
# n_rows = int(np.ceil(len(num_col) / n_cols))

# fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
# axes = axes.flatten()

# for i, col in enumerate(num_col):
#     sns.boxplot(y=x_train[col], ax=axes[i], color='skyblue')
#     axes[i].set_title(col)
#     axes[i].set_ylabel('')

# for j in range(i + 1, len(axes)):
#     fig.delaxes(axes[j])

# plt.tight_layout()
# plt.show()

--- Proportion of Outliers per Feature (IQR Method) ---
dttl                      : 23.64%
sttl                      : 23.61%
ct_dst_sport_ltm          : 22.45%
dload_log1p               : 19.72%
dmeansz_log1p             : 19.70%
dintpkt                   : 17.85%
sintpkt                   : 17.54%
dur_log1p                 : 16.83%
ct_dst_ltm                : 14.28%
ct_state_ttl              : 14.26%
ct_src_ltm                : 13.27%
ct_srv_src                : 12.35%
sload_log1p               : 10.88%
trans_depth               : 8.08%
ct_flw_http_mthd          : 8.08%
res_bdy_len               : 7.36%
smeansz_log1p             : 5.75%
ackdat                    : 4.94%
tcprtt                    : 4.42%
synack                    : 4.38%
ct_ftp_cmd                : 1.72%
is_ftp_login              : 1.71%
is_sm_ips_ports           : 0.16%
sbytes_log1p              : 0.06%
djit_log1p                : 0.02%
spkts_log1p               : 0.01%
dbytes_log1p              : 0.00%
network_bytes

## Interpreting the Outlier Proportions

The IQR results show high outlier rates for `dttl` (24%) and `sttl` (24%), but 0% for `dbytes_log1p`. This is mathematically correct but misleading for network data:

1. **Discrete Data:** Features like `dttl` are mostly 29. The IQR becomes tiny, flagging normal high values (like 252) as outliers.
2. **Log1p Effect:** The log transformation compressed extreme byte counts, hiding them from the IQR math.
3. **Binary Columns:** Columns like `is_ftp_login` are just 0s and 1s, confusing the IQR formula.

**Conclusion:** Do not drop these "outliers." In network traffic, outliers are often the actual attacks. Use `RobustScaler` or tree-based models (Random Forest/XGBoost) which handle them automatically.

### Z-Score Method
Finds values more than 3 standard deviations from the mean.

In [111]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Calculate Z-scores for all numerical columns
z_scores = np.abs(stats.zscore(x_train[num_col]))

# 2. Flag anything greater than 3 (99.7% of normal data falls within 3 std devs)
z_outliers_mask = z_scores > 3

# 3. Calculate the proportion (percentage) instead of count
z_outlier_proportions = pd.Series(z_outliers_mask.mean(axis=0), index=x_train[num_col].columns) * 100
z_outlier_proportions = z_outlier_proportions[z_outlier_proportions > 0].sort_values(ascending=False)

print("--- Proportion of Outliers per Feature (Z-Score > 3) ---")
for col, prop in z_outlier_proportions.items():
    print(f"{col:25s} : {prop:.2f}%")

--- Proportion of Outliers per Feature (Z-Score > 3) ---
smeansz_log1p             : 5.51%
dttl                      : 3.36%
ct_dst_ltm                : 2.98%
ct_src_ltm                : 2.90%
ct_dst_sport_ltm          : 2.79%
ct_srv_src                : 2.55%
dur_log1p                 : 2.32%
ct_ftp_cmd                : 1.72%
is_ftp_login              : 1.71%
tcprtt                    : 1.64%
synack                    : 1.42%
ackdat                    : 1.35%
res_bdy_len               : 0.76%
ct_flw_http_mthd          : 0.68%
sload_log1p               : 0.61%
sintpkt                   : 0.58%
ct_state_ttl              : 0.34%
dintpkt                   : 0.26%
trans_depth               : 0.22%
is_sm_ips_ports           : 0.16%
sbytes_log1p              : 0.07%
sjit_log1p                : 0.04%
spkts_log1p               : 0.01%
djit_log1p                : 0.01%
network_bytes_log1p       : 0.01%


## Z-Score vs IQR Outlier Detection

**Comparison:**
- **IQR:** Flagged 23.64% of `dttl` as outliers. Too strict for discrete data; over-flags normal traffic.
- **Z-Score:** Flagged only 3.36% of `dttl`. Too lenient; extreme values inflate the mean/std, hiding true outliers.
- **Both fail** on `log1p` features (0% outliers) because transformation compressed extremes.

**Conclusion:**
Do not drop these outliers. Attacks are extremes. 
Use **Tree-based models** (immune to outliers) or **RobustScaler** (distance-based models).

## Standardizing
    
As we have seen that the range of few features in this dataset is very large. So we will keep everything within certain range by applying standardscaler. After this all the features will have mean 0 and std 1

In [112]:
# Standardizing the data
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
scaler = StandardScaler()
scaler = scaler.fit(x_train[num_col])

In [113]:
x_train[num_col] = scaler.transform(x_train[num_col])

In [114]:
x_train.head()

,proto,state,sttl,dttl,service,swin,trans_depth,res_bdy_len,stime,sintpkt,...,sload_log1p,dload_log1p,spkts_log1p,stcpb_log1p,dtcpb_log1p,smeansz_log1p,dmeansz_log1p,sjit_log1p,djit_log1p,network_bytes_log1p
0,udp,INT,2.561444,-0.717760,others,-1.196045,-0.225343,-0.089113,-1.172764,-0.057567,...,-1.383776,-1.879995,-0.704801,-1.190007,-1.189697,-1.137341,-1.850553,0.204270,-0.899657,-1.229918
1,udp,INT,-0.037542,-0.717760,dns,-1.196045,-0.225343,-0.089113,0.868469,-0.069616,...,1.614925,-1.879995,-1.080734,-1.190007,-1.189697,0.565988,-1.850553,-0.995343,-0.899657,-1.062092
2,tcp,FIN,-0.426051,-0.041365,others,0.836096,-0.225343,-0.089113,-1.157401,-0.069485,...,0.153252,0.988351,0.843701,0.741629,0.888925,-0.582651,0.881301,-0.995343,0.170283,0.894089
3,tcp,FIN,-0.426051,-0.041365,ftp,0.836096,-0.225343,-0.089113,-1.136357,-0.063807,...,-1.071111,-0.033644,1.032630,0.895318,0.770420,-0.793357,-0.012242,1.251851,0.515730,0.318009
4,tcp,FIN,-0.426051,-0.041365,others,0.836096,-0.225343,-0.089113,-1.144197,-0.069180,...,-0.090531,0.478104,0.195816,0.787264,0.900555,0.059601,0.106012,0.337745,-0.495378,0.001507


## Onehot Encoding

In our dataset we have few categorical columns with text data.
But ML models can't process text data it can process numbers.

So we have to convert categorical columns to numerical columns in some way.
We will use onehotencoder where we will assign 1 if the value is present for the row and rest of the columns will be 0.

In [115]:
# Onehot Encoding
service_ = OneHotEncoder()
proto_ = OneHotEncoder()
state_ = OneHotEncoder()
ohe_service = service_.fit(x_train.service.values.reshape(-1,1))
ohe_proto = proto_.fit(x_train.proto.values.reshape(-1,1))
ohe_state = state_.fit(x_train.state.values.reshape(-1,1))

In [116]:
pieces = []
for col, ohe in zip(['proto', 'service', 'state'], [ohe_proto, ohe_service, ohe_state]):
    x = ohe.transform(x_train[col].values.reshape(-1, 1)).astype(np.uint8)  # sparse, 1 byte per value
    pieces.append(pd.DataFrame(
        x.toarray(),
        columns=[col + '_' + str(i) for i in ohe.categories_[0]],
        index=x_train.index
    ))
    x_train.drop(columns=col, inplace=True)   # free the original column right away
    del x
    gc.collect()

x_train = pd.concat([x_train] + pieces, axis=1)   # a single concat at the end
del pieces
gc.collect()

0

In [117]:
x_train.head()

,sttl,dttl,swin,trans_depth,res_bdy_len,stime,sintpkt,dintpkt,tcprtt,synack,...,state_INT,state_MAS,state_PAR,state_REQ,state_RST,state_TST,state_TXD,state_URH,state_URN,state_no
0,2.561444,-0.717760,-1.196045,-0.225343,-0.089113,-1.172764,-0.057567,-0.055099,-0.136439,-0.128893,...,1,0,0,0,0,0,0,0,0,0
1,-0.037542,-0.717760,-1.196045,-0.225343,-0.089113,0.868469,-0.069616,-0.055099,-0.136439,-0.128893,...,1,0,0,0,0,0,0,0,0,0
2,-0.426051,-0.041365,0.836096,-0.225343,-0.089113,-1.157401,-0.069485,-0.054857,-0.121383,-0.107394,...,0,0,0,0,0,0,0,0,0,0
3,-0.426051,-0.041365,0.836096,-0.225343,-0.089113,-1.136357,-0.063807,-0.044512,-0.120830,-0.107159,...,0,0,0,0,0,0,0,0,0,0
4,-0.426051,-0.041365,0.836096,-0.225343,-0.089113,-1.144197,-0.069180,-0.054358,-0.123351,-0.110421,...,0,0,0,0,0,0,0,0,0,0


Saving all the important parameters and objects to disk so that we can apply same process on test data

In [118]:
file_path = '.final_ipynb/'
os.makedirs(file_path, exist_ok=True)

In [119]:
pickle.dump(scaler, open(file_path+'scaler.pkl', 'wb'))  # Standard scaler

# Merge with the dictionary already on disk instead of replacing it:
# keys from this notebook refresh old values, keys that only exist on disk are kept
path = file_path + 'saved_dict.pkl'
if os.path.exists(path):
    with open(path, 'rb') as f:
        old_dict = pickle.load(f)
else:
    old_dict = {}

old_dict.update(saved_dict)
with open(path, 'wb') as f:
    pickle.dump(old_dict, f)  # Dictionary with important parameters

pickle.dump(mode_dict, open(file_path+'mode_dict.pkl', 'wb'))  # Dictionary with most frequent values of columns

In [120]:
# Onehot encoder for categorical columns
pickle.dump(ohe_proto, open(file_path+'ohe_proto.pkl', 'wb'))
pickle.dump(ohe_service, open(file_path+'ohe_service.pkl', 'wb'))
pickle.dump(ohe_state, open(file_path+'ohe_state.pkl', 'wb'))

In [121]:
# Final train/test pickles are saved after test preprocessing at the end of this notebook.

## Pipeline functions

We have to prepare a pipeline, where we can send raw data and get the output.

We will use test data to implement the pipeline. Here we will use all the parameters we have saved using train data.

Also standardize and onehot encode test data using train data objects for standardscaler and onehotencoder.

In [122]:
def clean_data(data):
    '''
    Cleans given raw data. Performs various cleaning, removes Null and wrong values.
    Check for columns datatype and fix them.
    '''
    numerical_col = data.select_dtypes(include=np.number).columns  # All the numerical columns list
    categorical_col = data.select_dtypes(exclude=np.number).columns  # All the categorical columns list
    
    # Cleaning the data
    for col in data.columns:
        val = mode_dict[col]  # Mode value of the column in train data
        data[col] = data[col].fillna(value=val)
        data[col] = data[col].replace(' ', value=val)
        data[col] = data[col].apply(lambda x:"None" if x=="-" else x)

        # Fixing binary columns
        if col in saved_dict['binary_col']:
            data[col] = np.where(data[col]>1, val, data[col])

    # Fixing datatype of columns
    bad_dtypes = list(set(categorical_col) - set(saved_dict['cat_col']))
    for bad_col in bad_dtypes:
        data[bad_col] = data[bad_col].astype(float)   # was data[col]: wrong column
    
    return data

In [123]:
def apply_log1p(data):
    '''
    Performs FE on the data. Apply log1p on the specified columns create new column and remove those original columns.
    '''
    for col in saved_dict['log1p_col']:
        new_col = col + '_log1p'  # New col name
        data[new_col] = data[col].apply(np.log1p)  # Creating new column on transformed data
        data.drop(col, axis=1, inplace=True)  # Removing old columns
    return data

In [124]:
def standardize(data):
    '''
    Stanardize the given data. Performs mean centering and varience scaling.
    Using stanardscaler object trained on train data.
    '''
    data[saved_dict['num_col']] = scaler.transform(data[saved_dict['num_col']])
    return data

In [125]:
def ohencoding(data):
    '''
    Onehot encoding the categorical columns.
    Add the ohe columns with the data and removes categorical columns.
    Using Onehotencoder objects trained on train data.
    '''
    pieces = []
    for col, ohe in [('proto', ohe_proto), ('service', ohe_service), ('state', ohe_state)]:
        X = ohe.transform(data[col].values.reshape(-1, 1)).astype(np.uint8)  # 1 byte per value
        pieces.append(pd.DataFrame(
            X.toarray(),
            columns=[col + '_' + str(c) for c in ohe.categories_[0]],
            index=data.index                      # same rows as data, avoids NaN rows
        ))

    data = data.drop(columns=['proto', 'service', 'state'])   # remove cat columns first
    return pd.concat([data] + pieces, axis=1)                 # a single concat

Loading all the objects from disk, that we have trained on train data.

In [126]:
# Parameters (the complete dictionary saved above)
saved_dict = pickle.load(open(file_path+'saved_dict.pkl', 'rb'))
# Mode value of all the columns
mode_dict = pickle.load(open(file_path+'mode_dict.pkl', 'rb'))
# Stanardscaler object
scaler = pickle.load(open(file_path+'scaler.pkl', 'rb'))

In [127]:
# One hot encoder objects
ohe_proto = pickle.load(open(file_path+'ohe_proto.pkl', 'rb'))
ohe_service = pickle.load(open(file_path+'ohe_service.pkl', 'rb'))
ohe_state = pickle.load(open(file_path+'ohe_state.pkl', 'rb'))

In [128]:
x_test.shape

(762015, 40)

In [129]:
# Resetting index of test data
x_test.reset_index(drop=True, inplace=True)

In [130]:
x_test.shape

(762015, 40)

In [131]:
x_test.columns

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'service', 'sload', 'dload', 'spkts', 'swin',
       'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len',
       'sjit', 'djit', 'stime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack',
       'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd',
       'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_dst_sport_ltm', 'attack_cat'],
      dtype='object')

`x_test` comes from `test_alldata_EDA.csv`, so it already has its column names. No renaming is needed here (`saved_dict['columns']` is only for raw files read with `header=None`).

In [132]:
# Creating new Feature
x_test['network_bytes'] = x_test['dbytes'] + x_test['sbytes']

In [133]:
# Droping all the unwanted columns (corr_col were already removed from test earlier, so skip missing ones)
dropable_col = saved_dict['to_drop'] + saved_dict['corr_col']
x_test.drop(columns=[c for c in dropable_col if c in x_test.columns], inplace=True)

In [134]:
x_test.shape

(762015, 36)

In [135]:
# Cleaning data using clean_data()
x_test = clean_data(x_test)

In [136]:
x_test.shape

(762015, 36)

In [137]:
# FE: applying log1p using apply_log1p()
x_test = apply_log1p(x_test)

In [138]:
x_test.shape

(762015, 36)

In [139]:
x_test.columns

Index(['proto', 'state', 'sttl', 'dttl', 'service', 'swin', 'trans_depth',
       'res_bdy_len', 'stime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack',
       'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd',
       'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_dst_sport_ltm', 'dur_log1p', 'sbytes_log1p', 'dbytes_log1p',
       'sload_log1p', 'dload_log1p', 'spkts_log1p', 'stcpb_log1p',
       'dtcpb_log1p', 'smeansz_log1p', 'dmeansz_log1p', 'sjit_log1p',
       'djit_log1p', 'network_bytes_log1p'],
      dtype='object')

In [140]:
# Standardscaling using stanardize()
x_test = standardize(x_test)

In [141]:
x_test.head()

,proto,state,sttl,dttl,service,swin,trans_depth,res_bdy_len,stime,sintpkt,...,sload_log1p,dload_log1p,spkts_log1p,stcpb_log1p,dtcpb_log1p,smeansz_log1p,dmeansz_log1p,sjit_log1p,djit_log1p,network_bytes_log1p
0,udp,CON,-0.426051,-0.041365,dns,-1.196045,-0.225343,-0.089113,-1.148407,-0.069615,...,-0.121716,0.482924,-1.080734,-1.190007,-1.189697,-0.374605,0.096502,-0.995343,-0.899657,-0.974789
1,udp,CON,-0.426051,-0.041365,dns,-1.196045,-0.225343,-0.089113,-1.163715,-0.069617,...,-0.112689,0.487911,-1.080734,-1.190007,-1.189697,-0.374605,0.096502,-0.995343,-0.899657,-0.974789
2,udp,INT,-0.037542,-0.717760,dns,-1.196045,-0.225343,-0.089113,0.855736,-0.069616,...,1.577442,-1.879995,-1.080734,-1.190007,-1.189697,0.565988,-1.850553,-0.995343,-0.899657,-1.062092
3,tcp,FIN,-0.426051,-0.041365,None,0.836096,-0.225343,-0.089113,-1.157963,-0.069518,...,0.619603,0.775428,0.277671,0.898135,0.898496,1.366104,0.278308,-0.163891,-0.794201,0.298473
4,udp,CON,-0.426051,-0.041365,dns,-1.196045,-0.225343,-0.089113,0.864749,-0.069616,...,-0.150173,0.467203,-1.080734,-1.190007,-1.189697,-0.374605,0.096502,-0.995343,-0.899657,-0.974789


In [142]:
x_test['service'] = x_test['service'].replace('None', 'others')

for col, ohe in [('proto', ohe_proto), ('service', ohe_service), ('state', ohe_state)]:
    print(col, set(x_test[col].unique()) - set(ohe.categories_[0]))   # each set should be empty

x_test = ohencoding(x_test)

proto set()
service set()
state set()


In [143]:
# Onehot encoding categorical columns using ohencoding()
# x_test = ohencoding(x_test)

# x_train.filter(regex='^proto_').columns

# binary_cols = [c for c in x_train.columns if x_train[c].dropna().isin([0, 1]).all()]
# print(binary_cols)
saved_dict['binary_col']
x_train[saved_dict['binary_col']]

,is_sm_ips_ports,is_ftp_login
0,-0.04053,-0.132020
1,-0.04053,-0.132020
2,-0.04053,-0.132020
3,-0.04053,7.574602
4,-0.04053,-0.132020
...,...,...
1778027,-0.04053,-0.132020
1778028,-0.04053,-0.132020
1778029,-0.04053,-0.132020
1778030,-0.04053,-0.132020


In [144]:
x_test.shape

(762015, 197)

In [145]:
# Final test data
x_test.head()

,sttl,dttl,swin,trans_depth,res_bdy_len,stime,sintpkt,dintpkt,tcprtt,synack,...,state_INT,state_MAS,state_PAR,state_REQ,state_RST,state_TST,state_TXD,state_URH,state_URN,state_no
0,-0.426051,-0.041365,-1.196045,-0.225343,-0.089113,-1.148407,-0.069615,-0.055095,-0.136439,-0.128893,...,0,0,0,0,0,0,0,0,0,0
1,-0.426051,-0.041365,-1.196045,-0.225343,-0.089113,-1.163715,-0.069617,-0.055093,-0.136439,-0.128893,...,0,0,0,0,0,0,0,0,0,0
2,-0.037542,-0.717760,-1.196045,-0.225343,-0.089113,0.855736,-0.069616,-0.055099,-0.136439,-0.128893,...,1,0,0,0,0,0,0,0,0,0
3,-0.426051,-0.041365,0.836096,-0.225343,-0.089113,-1.157963,-0.069518,-0.054912,-0.121029,-0.107473,...,0,0,0,0,0,0,0,0,0,0
4,-0.426051,-0.041365,-1.196045,-0.225343,-0.089113,0.864749,-0.069616,-0.055095,-0.136439,-0.128893,...,0,0,0,0,0,0,0,0,0,0


In [146]:
# Matching test data columns with train data columns
all(x_train.columns == x_test.columns)

True

In [147]:
# Save exactly the objects loaded by ML Models and Results.ipynb.
assert x_train.columns.equals(x_test.columns), "Train/test feature columns do not match."
assert len(x_train) == len(y_train), "Training features and labels have different row counts."
assert len(x_test) == len(y_test), "Test features and labels have different row counts."
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in x_train.dtypes), "Model features must all be numeric."

os.makedirs(file_path, exist_ok=True)
for split_name, features, target in [
    ('final_train.pkl', x_train, y_train),
    ('final_test.pkl', x_test, y_test),
]:
    output_path = os.path.join(file_path, split_name)
    temporary_path = output_path + '.tmp'
    with open(temporary_path, 'wb') as output_file:
        pickle.dump((features, target), output_file, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temporary_path, output_path)
    print(f"Saved {output_path}: features={features.shape}, labels={target.shape}")

Saved .final_ipynb/final_train.pkl: features=(1778032, 197), labels=(1778032,)
Saved .final_ipynb/final_test.pkl: features=(762015, 197), labels=(762015,)
